In [1]:
import os
import numpy as np
import nibabel as nib
from nilearn.input_data import NiftiLabelsMasker
from nilearn.image import clean_img
import pandas as pd
from scipy.io import loadmat
import pandas as pd
from nilearn.image import clean_img, index_img
from nilearn.input_data import NiftiLabelsMasker


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\805655390.py:4: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  from nilearn.input_data import NiftiLabelsMasker


Load in Shen atlas and define NiftiMasker used for transforming fMRI voxel space data into parcellated ROI space

In [2]:
# Define paths
shen_atlas_path = "../3_atlases/Shen/shen_2mm_268_parcellation.nii.gz"  

# Initialize the masker with the Shen atlas
masker = NiftiLabelsMasker(labels_img=shen_atlas_path)


Compute dFC for EEG Rest

In [3]:
# Set paths
fmri_dir = "../1_data/1_EEG_Rest_fMRI/"  

# Define TR and target FC window duration
TR = 2.1  # Repetition Time (seconds)
target_window_seconds = 30  # Desired window duration in seconds

# Compute window size in TRs
window_size = max(1, int(target_window_seconds / TR))  # Ensure at least 1 TR
print(f"Window size: {window_size}")

# Define Gaussian window parameters
sigma = window_size / 4  # Controls decay of Gaussian weights
center = window_size // 2  # Midpoint of the window

# Generate Gaussian weights
indices = np.arange(window_size)
weights = np.exp(-((indices - center) ** 2) / (2 * sigma ** 2))
weights /= np.sum(weights)  # Normalize weights

# Get a list of unique subjects from filenames
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])

fc_list = []
# Process each subject
for file in fmri_files:
    subj = file.split("_")[0]
    run = ''.join([char for char in file.split("_")[2] if char.isdigit()])
    fmri_file = file
    confound_file = f"{subj}_task-rest_run-{run}_desc-confounds_timeseries.tsv"

    fmri_path = os.path.join(fmri_dir, fmri_file)
    confound_path = os.path.join(fmri_dir, confound_file)

    # Skip if either file is missing
    if not os.path.exists(fmri_path) or not os.path.exists(confound_path):
        print(f"Missing file for {subj} run-{run}, skipping.")
        continue

    print(f"\nProcessing: {fmri_file}")

    # 📌 **Step 1: Load & Prepare Confounds**
    confounds = pd.read_csv(confound_path, sep="\t")

    # Select confounds to regress out
    confound_columns = ["global_signal", "white_matter", "csf",
                        "trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]


    # 📌 **Check for Excessive Motion**
    motion_threshold_mm = 3.0  # Max allowed translation in mm
    rotation_threshold_rad = 3 * (np.pi / 180)  # Convert 3 degrees to radians

    # Extract motion parameters
    motion_params = confounds[["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]].copy()

    # Compute max absolute motion per run
    max_translation = motion_params[["trans_x", "trans_y", "trans_z"]].abs().max().max()
    max_rotation = motion_params[["rot_x", "rot_y", "rot_z"]].abs().max().max()

    # Flag runs with excessive motion
    if max_translation > motion_threshold_mm or max_rotation > rotation_threshold_rad:
        print(f"⚠️ WARNING: {subj} run-{run} exceeded motion threshold!")
        print(f"   Max Translation: {max_translation:.2f} mm")
        print(f"   Max Rotation: {np.degrees(max_rotation):.2f}°")

    # Ensure selected confounds exist in the file
    confound_matrix = confounds[confound_columns].copy()

    cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)

    # Extract ROI time series using masker
    time_series = masker.fit_transform(X=cleaned_img)
    n_TRs, n_regions = time_series.shape

    # Initialize array to store FC vectors per TR
    fc_vectors = np.full((n_TRs, (n_regions * (n_regions - 1)) // 2), np.nan)

    # Compute FC using a Gaussian-weighted sliding window
    for tr in range(n_TRs):
        if tr < center or tr >= (n_TRs - center):
            continue  # Skip edge TRs

        # Define window range correctly
        start = max(0, tr - center)
        end = min(n_TRs, tr + center)  # REMOVE +1
        
        # Extract time series within the window
        ts_window = time_series[start:end]
        
        # Trim Gaussian weights dynamically
        effective_weights = weights[: ts_window.shape[0]]
        effective_weights /= np.sum(effective_weights)  # Normalize weights
        
        # Apply Gaussian weights (Now the shapes will always match)
        weighted_ts = ts_window * effective_weights[:, np.newaxis]


        # Compute weighted covariance matrix
        weighted_cov = np.cov(weighted_ts.T, aweights=weights)

        # Convert covariance to correlation
        std_dev = np.sqrt(np.diag(weighted_cov))

        # Prevent division by zero by replacing zeros with a small epsilon
        std_dev[std_dev == 0] = 1e-6  # Small value to avoid NaN

        fc_matrix = weighted_cov / np.outer(std_dev, std_dev)

        # Extract upper triangle as feature vector
        fc_vector = fc_matrix[np.triu_indices_from(fc_matrix, k=1)]

        # Store in matrix
        fc_vectors[tr] = fc_vector

    # Store FC vectors for this run
    fc_list.append(fc_vectors)

# Convert list to NumPy array
fc_array = np.array(fc_list)  # Shape: (n_subjects, 2, n_TRs, n_features)

# Compute number of subjects and features
n_files, n_TRs, n_features = fc_array.shape
n_subjects = n_files // 2  # Since each subject has exactly two runs

# Reshape the array to (n_subjects, 2, n_features)
fc_array_reshaped = fc_array.reshape(n_subjects, 2, n_TRs, n_features)

# Save reshaped array
np.save("../4_results/1_arousal/1_EEG_Rest/EEG_FC_array.npy", fc_array_reshaped)
print(f"✅ Saved FC array with shape: {fc_array_reshaped.shape}")


Window size: 14

Processing: sub-02_task-rest_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc


Processing: sub-02_task-rest_run2_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc


Processing: sub-03_task-rest_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc

... (95 outputs left out)

C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc


Processing: sub-33_task-rest_run2_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3367708137.py:70: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc

✅ Saved FC array with shape: (26, 2, 286, 35778)


Compute dFC for Pupil Rest

In [4]:
# Set paths
fmri_dir = "../1_data/2_Pupil_Rest_fMRI/"  # Update with actual path

# Define TR and target FC window duration
TR = 1  # Repetition Time (seconds)
target_window_seconds = 30  # Desired window duration in seconds

# Compute window size in TRs
window_size = max(1, int(target_window_seconds / TR))  # Ensure at least 1 TR
print(f"Window size: {window_size}")

# Define Gaussian window parameters
sigma = window_size / 4  # Controls decay of Gaussian weights
center = window_size // 2  # Midpoint of the window

# Generate Gaussian weights
indices = np.arange(window_size)
weights = np.exp(-((indices - center) ** 2) / (2 * sigma ** 2))
weights /= np.sum(weights)  # Normalize weights

# Get a list of unique subjects from filenames
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])

fc_list = []
# Process each subject
for file in fmri_files:
    subj = file.split("_")[0]
    run = ''.join([char for char in file.split("_")[2] if char.isdigit()])
    fmri_file = file
    confound_file = f"{subj}_task-rest_run-0{run}_desc-confounds_timeseries.tsv"
    #print(confound_file)

    fmri_path = os.path.join(fmri_dir, fmri_file)
    confound_path = os.path.join(fmri_dir, confound_file)

    # Skip if either file is missing
    if not os.path.exists(fmri_path) or not os.path.exists(confound_path):
        print(f"Missing file for {subj} run-{run}, skipping.")
        continue

    print(f"\nProcessing: {fmri_file}")

    # 📌 **Step 2: Load & Trim Confounds**
    confounds = pd.read_csv(confound_path, sep="\t").reset_index(drop=True)

    # Select confounds to regress out
    confound_columns = ["global_signal", "white_matter", "csf",
                        "trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]  # Other confounds can be added if needed
    confound_matrix = confounds[confound_columns].copy()

    # 📌 **Check for Excessive Motion**
    motion_threshold_mm = 3.0  # Maximum allowed translation in mm
    rotation_threshold_rad = 3 * (np.pi / 180)  # Convert 3 degrees to radians
    
    # Extract motion parameters
    motion_params = confounds[["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]].copy()
    
    # Compute max absolute motion per run
    max_translation = motion_params[["trans_x", "trans_y", "trans_z"]].abs().max().max()
    max_rotation = motion_params[["rot_x", "rot_y", "rot_z"]].abs().max().max()

    # Flag runs with excessive motion
    if max_translation > motion_threshold_mm or max_rotation > rotation_threshold_rad:
        print(f"⚠️ WARNING: {subj} run-{run} exceeded motion threshold!")
        print(f"   Max Translation: {max_translation:.2f} mm")
        print(f"   Max Rotation: {np.degrees(max_rotation):.2f}°")

    cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)

    # 📌 **Step 7: Extract ROI Time Series**
    time_series = masker.fit_transform(cleaned_img)[10:,:]
    n_TRs, n_regions = time_series.shape
    print(n_TRs, n_regions)

    # 📌 **Step 8: Compute Functional Connectivity (FC)**
    fc_vectors = np.full((n_TRs, (n_regions * (n_regions - 1)) // 2), np.nan)

    for tr in range(n_TRs):
        if tr < center or tr >= (n_TRs - center):
            continue  # Skip edge TRs

        # Define window range dynamically
        start = max(0, tr - center)
        end = min(n_TRs, tr + center)

        # Extract time series within the window
        ts_window = time_series[start:end]

        # Apply Gaussian weights (handle cases where window is shorter at edges)
        effective_weights = weights[: ts_window.shape[0]]
        effective_weights /= np.sum(effective_weights)  # Normalize weights

        weighted_ts = ts_window * effective_weights[:, np.newaxis]

        # Compute weighted covariance matrix
        weighted_cov = np.cov(weighted_ts.T, aweights=effective_weights)

        # Convert covariance to correlation
        std_dev = np.sqrt(np.diag(weighted_cov))

        # Prevent division by zero by replacing zeros with a small epsilon
        std_dev[std_dev == 0] = 1e-6  

        fc_matrix = weighted_cov / np.outer(std_dev, std_dev)

        # Extract upper triangle as feature vector
        fc_vector = fc_matrix[np.triu_indices_from(fc_matrix, k=1)]

        # Store in matrix
        fc_vectors[tr] = fc_vector

    # Store FC vectors for this run
    fc_list.append(fc_vectors)

# 📌 **Step 9: Save Processed Data**
fc_array = np.array(fc_list)  # Shape: (n_subjects, 2, n_TRs, n_features)

# Compute number of subjects and features
n_files, n_TRs, n_features = fc_array.shape
n_subjects = n_files // 2  # Since each subject has exactly two runs

# Reshape the array to (n_subjects, 2, n_TRs, n_features)
fc_array_reshaped = fc_array.reshape(n_subjects, 2, n_TRs, n_features)

# Save reshaped array
output_path = "../4_results/1_arousal/2_Pupil_Rest/Pupil_FC_array.npy"
np.save(output_path, fc_array_reshaped)
print(f"✅ Saved FC array with shape: {fc_array_reshaped.shape} at {output_path}")


Window size: 30
sub-pa1372_task-rest_run-01_desc-confounds_timeseries.tsv

Processing: sub-pa1372_task-rest_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc

400 268
sub-pa1372_task-rest_run-02_desc-confounds_timeseries.tsv

Processing: sub-pa1372_task-rest_run2_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc

400 268
sub-pa1387_task-rest_run-01_desc-confounds_timeseries.tsv

Processing: sub-pa1387_task-rest_run1_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc

... (99 outputs left out)

C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc

400 268
sub-pa2226_task-rest_run-02_desc-confounds_timeseries.tsv

Processing: sub-pa2226_task-rest_run2_space-MNI152NLin6Asym_desc-filtered_bold.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\1167615006.py:68: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zsc

400 268
✅ Saved FC array with shape: (27, 2, 400, 35778) at ../4_results/1_arousal/2_Pupil_Rest/Pupil_FC_array.npy


Compute dFC for Pupil Story

In [5]:
# Set paths
fmri_dir = "../1_data/3_Pupil_Story_fMRI/"  # Update with actual path

# Define TR and target FC window duration
TR = 1  # Repetition Time (seconds)
target_window_seconds = 30  # Desired window duration in seconds

# Compute window size in TRs
window_size = max(1, int(target_window_seconds / TR))  # Ensure at least 1 TR
print(f"Window size: {window_size}")

# Define Gaussian window parameters
sigma = window_size / 4  # Controls decay of Gaussian weights
center = window_size // 2  # Midpoint of the window

# Generate Gaussian weights
indices = np.arange(window_size)
weights = np.exp(-((indices - center) ** 2) / (2 * sigma ** 2))
weights /= np.sum(weights)  # Normalize weights

# Get a list of unique subjects from filenames
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])

fc_list = []
# Process each subject
for file in fmri_files:
    subj = file.split("_")[0]
    # Extract the part containing "story1"
    task_part = [part for part in file.split("_") if "story" in part][0]
    
    # Extract the number from "story1"
    run = int("".join([char for char in task_part if char.isdigit()]))
    fmri_file = file
    print(f"On file: {fmri_file}, which is subj {subj} and run: {run}")
    confound_file = f"{subj}_task-story{run}_desc-confounds_timeseries.tsv"
    #print(confound_file)

    fmri_path = os.path.join(fmri_dir, fmri_file)
    confound_path = os.path.join(fmri_dir, confound_file)

    # Skip if either file is missing
    if not os.path.exists(fmri_path) or not os.path.exists(confound_path):
        print(f"Missing file for {subj} run-{run}, skipping.")
        continue

    print(f"\nProcessing: {fmri_file}")

    # 📌 **Step 2: Load & Trim Confounds**
    confounds = pd.read_csv(confound_path, sep="\t").reset_index(drop=True)

    # Select confounds to regress out
    confound_columns = ["global_signal", "white_matter", "csf",
                        "trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]  # Other confounds can be added if needed
    confound_matrix = confounds[confound_columns].copy()

    # 📌 **Check for Excessive Motion**
    motion_threshold_mm = 3.0  # Maximum allowed translation in mm
    rotation_threshold_rad = 3 * (np.pi / 180)  # Convert 3 degrees to radians
    
    # Extract motion parameters
    motion_params = confounds[["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]].copy()
    
    # Compute max absolute motion per run
    max_translation = motion_params[["trans_x", "trans_y", "trans_z"]].abs().max().max()
    max_rotation = motion_params[["rot_x", "rot_y", "rot_z"]].abs().max().max()

    # Flag runs with excessive motion
    if max_translation > motion_threshold_mm or max_rotation > rotation_threshold_rad:
        print(f"⚠️ WARNING: {subj} run-{run} exceeded motion threshold!")
        print(f"   Max Translation: {max_translation:.2f} mm")
        print(f"   Max Rotation: {np.degrees(max_rotation):.2f}°")

    cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)

    # 📌 **Step 7: Extract ROI Time Series**
    if run == 1:
        time_series = masker.fit_transform(cleaned_img)[2:,:]
    elif run == 2:
       time_series = masker.fit_transform(cleaned_img)[3:,:] 
    elif run == 3:
        time_series = masker.fit_transform(cleaned_img)[3:,:]
    else:
        print("No run number")
    n_TRs, n_regions = time_series.shape
    print(n_TRs, n_regions)

    # 📌 **Step 8: Compute Functional Connectivity (FC)**
    fc_vectors = np.full((n_TRs, (n_regions * (n_regions - 1)) // 2), np.nan)

    for tr in range(n_TRs):
        if tr < center or tr >= (n_TRs - center):
            continue  # Skip edge TRs

        # Define window range dynamically
        start = max(0, tr - center)
        end = min(n_TRs, tr + center)

        # Extract time series within the window
        ts_window = time_series[start:end]

        # Apply Gaussian weights (handle cases where window is shorter at edges)
        effective_weights = weights[: ts_window.shape[0]]
        effective_weights /= np.sum(effective_weights)  # Normalize weights

        weighted_ts = ts_window * effective_weights[:, np.newaxis]

        # Compute weighted covariance matrix
        weighted_cov = np.cov(weighted_ts.T, aweights=effective_weights)

        # Convert covariance to correlation
        std_dev = np.sqrt(np.diag(weighted_cov))

        # Prevent division by zero by replacing zeros with a small epsilon
        std_dev[std_dev == 0] = 1e-6  

        fc_matrix = weighted_cov / np.outer(std_dev, std_dev)

        # Extract upper triangle as feature vector
        fc_vector = fc_matrix[np.triu_indices_from(fc_matrix, k=1)]

        # Store in matrix
        fc_vectors[tr] = fc_vector

    # Store FC vectors for this run
    fc_list.append(fc_vectors)

# Number of subjects (since we have 3 runs per subject)
n_subjects = len(fc_list) // 3  # Assuming each subject has exactly 3 runs

# Ensure that fc_list is divisible by 3
if len(fc_list) % 3 != 0:
    raise ValueError("Mismatch in fc_list structure! Ensure every subject has exactly 3 runs.")

# Reshape into (subjects, runs)
fc_array_per_run = np.array(fc_list, dtype=object).reshape(n_subjects, 3)  # (n_subjects, 3)

# Process each subject
subject_arrays = []
for subj_idx in range(n_subjects):
    runs = fc_array_per_run[subj_idx]  # Get all 3 runs for the subject

    trimmed_runs = []
    for run_idx, fc_vectors in enumerate(runs):
        if fc_vectors.shape[0] > 30:  # Ensure enough TRs to trim
            trimmed_fc = fc_vectors[15:-15, :]  # Trim first 15 and last 15 TRs
        else:
            print(f"⚠️ Warning: Subject {subj_idx+1} Run {run_idx+1} has fewer than 30 TRs. Skipping trimming.")
            trimmed_fc = fc_vectors  # Keep unchanged if too short
        
        trimmed_runs.append(trimmed_fc)

    # Concatenate runs in order: run1 → run2 → run3
    full_fc = np.concatenate(trimmed_runs, axis=0)  # Shape: (total_n_TRs, n_features)
    subject_arrays.append(full_fc)

# Convert to NumPy array (n_subjects, total_n_TRs, n_features)
fc_array_final = np.array(subject_arrays)

# Save final FC array
output_path = "../4_results/1_arousal/3_Pupil_Story/Pupil_Story_FC_array.npy"
np.save(output_path, fc_array_final)

print(f"✅ Saved FC array with shape: {fc_array_final.shape} at {output_path}")


Window size: 30
On file: sub-tb2994_task-story1_filtered.nii.gz, which is subj sub-tb2994 and run: 1

Processing: sub-tb2994_task-story1_filtered.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

524 268
On file: sub-tb2994_task-story2_filtered.nii.gz, which is subj sub-tb2994 and run: 2

Processing: sub-tb2994_task-story2_filtered.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

449 268
On file: sub-tb2994_task-story3_filtered.nii.gz, which is subj sub-tb2994 and run: 3

Processing: sub-tb2994_task-story3_filtered.nii.gz
⚠️ WARNING: sub-tb2994 run-3 exceeded motion threshold!
   Max Translation: 1.89 mm
   Max Rotation: 4.09°


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

... (123 outputs left out)

C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

449 268
On file: sub-tb4572_task-story3_filtered.nii.gz, which is subj sub-tb4572 and run: 3

Processing: sub-tb4572_task-story3_filtered.nii.gz


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend = False, standardize = True, confounds = confound_matrix)
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\606781901.py:73: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore

329 268
✅ Saved FC array with shape: (22, 1212, 35778) at ../4_results/1_arousal/3_Pupil_Story/Pupil_Story_FC_array.npy


Compute dFC for FilmFest Event Segments

In [6]:
# -----------------------------
# CONFIGURATION
# -----------------------------
fmri_dir = "../1_data/4_FilmFestival_fMRI/"
event_file = "../1_data/4_FilmFestival_Arousal/Filmfest_timestamps.xlsx"  
output_path = "../4_results/1_arousal/4_FilmFestival/FilmFestival_FC_array.npy"

TR = 1.5
trim_front = 5  # remove first 5 TRs from each run

# -----------------------------
# LOAD EVENTS 
# -----------------------------
events_df = pd.read_excel(event_file)
n_events_per_run = len(events_df)

# -----------------------------
# LOAD AND PROCESS EACH RUN
# -----------------------------
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])
fc_list = []

for file in fmri_files:
    parts = file.split("_")
    subj = parts[0]
    run_part = [p for p in parts if p.startswith("run") and "-" not in p][0]
    run = int(run_part.replace("run", ""))
    is_run1 = run == 1

    # Event bounds for this run
    start_col = "Movie1_TR_start" if is_run1 else "Movie2_TR_start"
    end_col = "Movie1_TR_end" if is_run1 else "Movie2_TR_end"
    # Drop rows with NaNs in either TR column
    event_bounds = events_df[[start_col, end_col]].dropna().values
    events_df["Movie1_TR_start"] = events_df["Movie1_TR_start"].astype("Int64")  # Nullable integer type
    events_df["Movie1_TR_end"]   = events_df["Movie1_TR_end"].astype("Int64")
    events_df["Movie2_TR_start"] = events_df["Movie2_TR_start"].astype("Int64")
    events_df["Movie2_TR_end"]   = events_df["Movie2_TR_end"].astype("Int64")


    # File paths
    fmri_path = os.path.join(fmri_dir, file)
    confound_file = f"{subj}_task-movie_run-{run:02d}_desc-confounds_timeseries.tsv"
    confound_path = os.path.join(fmri_dir, confound_file)

    if not os.path.exists(fmri_path) or not os.path.exists(confound_path):
        print(f"❌ Missing file for {subj} run-{run}, skipping.")
        continue

    print(f"\n▶️ Processing {subj}, run-{run}")

    # Load image and confounds
    confounds = pd.read_csv(confound_path, sep="\t").reset_index(drop=True)


    # Motion QC
    confound_columns = [
        "global_signal", "white_matter", "csf",
        "trans_x", "trans_y", "trans_z",
        "rot_x", "rot_y", "rot_z"
    ]
    motion_threshold_mm = 3.0
    rotation_threshold_rad = 3 * (np.pi / 180)
    motion_params = confounds[["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]]
    max_translation = motion_params[["trans_x", "trans_y", "trans_z"]].abs().max().max()
    max_rotation = motion_params[["rot_x", "rot_y", "rot_z"]].abs().max().max()
    if max_translation > motion_threshold_mm or max_rotation > rotation_threshold_rad:
        print(f"⚠️ Motion warning for {subj} run-{run}: max translation: {max_translation}, max rotation: {max_rotation}")

    # Clean image
    cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
    time_series = masker.fit_transform(cleaned_img)

    # Trim first 5 TRs
    if time_series.shape[0] <= trim_front:
        print(f"⚠️ Not enough TRs after trim, skipping {subj} run-{run}")
        continue
    time_series = time_series[trim_front:]
    n_TRs, n_regions = time_series.shape
    n_features = (n_regions * (n_regions - 1)) // 2

    # Compute FC per event
    event_fc_vectors = []
    for start_tr, end_tr in event_bounds:
        start = start_tr 
        end = end_tr  + 1  # inclusive

        if start < 0:
            print(f"⚠️ Event starts before time series: {start}-{end}")
            event_alertness.append(np.nan)
            continue

        if end > n_TRs:
            print(f"⚠️ Event ends beyond time series: {start}-{end} → truncating")
            end = n_TRs

        # if start < 0 or end > n_TRs:
        #     print(f"⚠️ Event out of bounds: {start_tr}-{end_tr}")
        #     event_fc_vectors.append(np.full(n_features, np.nan))
        #     continue

        ts = time_series[start:end]
        if ts.shape[0] < 2:
            print("Error: not enough events")
            event_fc_vectors.append(np.full(n_features, np.nan))
            continue

        cov = np.cov(ts.T)
        std = np.sqrt(np.diag(cov))
        std[std == 0] = 1e-6
        corr = cov / np.outer(std, std)
        fc_vec = corr[np.triu_indices_from(corr, k=1)]
        event_fc_vectors.append(fc_vec)

    fc_list.append(np.stack(event_fc_vectors))  # shape: (n_events, n_features)

# -----------------------------
# GROUP BY SUBJECT, CONCATENATE RUNS
# -----------------------------
if len(fc_list) % 2 != 0:
    raise ValueError("⚠️ Each subject must have exactly 2 runs.")

n_subjects = len(fc_list) // 2
fc_array_per_run = np.array(fc_list, dtype=object).reshape(n_subjects, 2)
subject_arrays = []

for subj_idx in range(n_subjects):
    run1 = fc_array_per_run[subj_idx, 0]
    run2 = fc_array_per_run[subj_idx, 1]


    subject_fc = np.concatenate([run1, run2], axis=0)  # shape: (2*n_events, n_features)
    subject_arrays.append(subject_fc)

fc_array_final = np.stack(subject_arrays)  # shape: (n_subjects, 2*n_events, n_features)

# -----------------------------
# SAVE FINAL OUTPUT
# -----------------------------
np.save(output_path, fc_array_final)
print(f"\n✅ Saved final FC array: {fc_array_final.shape} → {output_path}")


▶️ Processing sub-01, run-1


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate


▶️ Processing sub-01, run-2


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate


▶️ Processing sub-02, run-1


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate

... (71 outputs left out)

C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate


▶️ Processing sub-20, run-2


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\2376705718.py:71: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate


✅ Saved final FC array: (20, 68, 35778) → ../4_results/1_arousal/4_FilmFestival/FilmFestival_FC_array.npy


Compute dFC for Sherlock Event Segmentation

In [7]:
# -----------------------------
# CONFIGURATION
# -----------------------------
fmri_dir = "../1_data/5_Sherlock_fMRI/"
event_file = "../1_data/5_Sherlock_Arousal/Sherlock_timestamps.xlsx"
output_path = "../4_results/1_arousal/5_Sherlock/Sherlock_FC_array.npy"

TR = 1.5
trim_front = 5  # Remove first 5 TRs from each run


# -----------------------------
# LOAD EVENTS
# -----------------------------
events_df = pd.read_excel(event_file)
events_df["run1_start"] = events_df["run1_start"].astype("Int64")
events_df["run1_end"] = events_df["run1_end"].astype("Int64")
events_df["run2_start"] = events_df["run2_start"].astype("Int64")
events_df["run2_end"] = events_df["run2_end"].astype("Int64")

# -----------------------------
# LOAD AND PROCESS EACH RUN
# -----------------------------
fmri_files = sorted([f for f in os.listdir(fmri_dir) if f.endswith(".nii.gz")])
fc_list = []

for file in fmri_files:
    parts = file.split("_")
    subj = parts[0]
    run_part = parts[2]  # should be part1 or part2
    run = 1 if "part1" in run_part else 2
    is_run1 = run == 1

    # Event bounds for this run
    start_col = "run1_start" if is_run1 else "run2_start"
    end_col = "run1_end" if is_run1 else "run2_end"
    event_bounds = events_df[[start_col, end_col]].dropna().values

    # File paths
    fmri_path = os.path.join(fmri_dir, file)
    confound_file = f"{subj}_task-sherlockPart{run}_desc-confounds_timeseries.tsv"
    confound_path = os.path.join(fmri_dir, confound_file)

    if not os.path.exists(fmri_path) or not os.path.exists(confound_path):
        print(f"❌ Missing file for {subj} part-{run}, skipping.")
        continue

    print(f"\n▶️ Processing {subj}, part-{run}")

    # Load image and confounds
    confounds = pd.read_csv(confound_path, sep="\t").reset_index(drop=True)

    # Trim confound rows
    if subj == "sub-04":
        start_idx = 32
        end_idx = 952 if is_run1 else 1036
    else:
        start_idx = 46
        end_idx = 966 if is_run1 else 1050

    if confounds.shape[0] < end_idx:
        print(f"⚠️ Confound file too short for {subj} part-{run}, skipping.")
        continue

    confounds = confounds.iloc[start_idx:end_idx].reset_index(drop=True)

    # Clean image
    confound_columns = [
        "global_signal", "white_matter", "csf",
        "trans_x", "trans_y", "trans_z",
        "rot_x", "rot_y", "rot_z"
    ]
    
    motion_threshold_mm = 3.0
    rotation_threshold_rad = 3 * (np.pi / 180)
    motion_params = confounds[["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]]
    max_translation = motion_params[["trans_x", "trans_y", "trans_z"]].abs().max().max()
    max_rotation = motion_params[["rot_x", "rot_y", "rot_z"]].abs().max().max()
    if max_translation > motion_threshold_mm or max_rotation > rotation_threshold_rad:
        print(f"⚠️ Motion warning for {subj} run-{run}: max translation: {max_translation}, max rotation: {max_rotation}")
        
    cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
    time_series = masker.fit_transform(cleaned_img)

    # Trim first 5 TRs
    if time_series.shape[0] <= trim_front:
        print(f"⚠️ Not enough TRs after trim, skipping {subj} part-{run}")
        continue
    time_series = time_series[trim_front:]
    n_TRs, n_regions = time_series.shape
    n_features = (n_regions * (n_regions - 1)) // 2
    print(n_TRs)

    # Compute FC per event
    event_fc_vectors = []
    for start_tr, end_tr in event_bounds:
        start = start_tr 
        end = end_tr 

        if start < 0:
            print(f"⚠️ Event starts before time series: {start}-{end}")
            event_alertness.append(np.nan)
            continue

        if end > n_TRs:
            print(f"⚠️ Event ends beyond time series: {start}-{end} → truncating")
            end = n_TRs


        ts = time_series[start:end]
        if ts.shape[0] < 2:
            event_fc_vectors.append(np.full(n_features, np.nan))
            continue

        cov = np.cov(ts.T)
        std = np.sqrt(np.diag(cov))
        std[std == 0] = 1e-6
        corr = cov / np.outer(std, std)
        fc_vec = corr[np.triu_indices_from(corr, k=1)]
        event_fc_vectors.append(fc_vec)

    fc_list.append(np.stack(event_fc_vectors))  # shape: (n_events, n_features)

# -----------------------------
# GROUP BY SUBJECT, CONCATENATE RUNS
# -----------------------------
if len(fc_list) % 2 != 0:
    raise ValueError("⚠️ Each subject must have exactly 2 runs.")

n_subjects = len(fc_list) // 2
fc_array_per_run = np.array(fc_list, dtype=object).reshape(n_subjects, 2)
subject_arrays = []

for subj_idx in range(n_subjects):
    run1 = fc_array_per_run[subj_idx, 0]
    run2 = fc_array_per_run[subj_idx, 1]


    subject_fc = np.concatenate([run1, run2], axis=0)  # shape: (2*n_events, n_features)
    subject_arrays.append(subject_fc)

fc_array_final = np.stack(subject_arrays)  # shape: (n_subjects, 2*n_events, n_features)

# -----------------------------
# SAVE FINAL OUTPUT
# -----------------------------
np.save(output_path, fc_array_final)
print(f"\n✅ Saved final FC array: {fc_array_final.shape} → {output_path}")


▶️ Processing sub-01, part-1


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate

915
⚠️ Event ends beyond time series: 866-919 → truncating

▶️ Processing sub-01, part-2


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate

999
⚠️ Event ends beyond time series: 975-1004 → truncating

▶️ Processing sub-02, part-1


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate

... (51 outputs left out)

C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate

915
⚠️ Event ends beyond time series: 866-919 → truncating

▶️ Processing sub-16, part-2


C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: DeprecationWarning: From release 0.14.0, confounds will be standardized using the sample std instead of the population std.
  cleaned_img = clean_img(fmri_path, detrend=False, standardize=True, confounds=confounds[confound_columns])
C:\Users\kanno\AppData\Local\Temp\ipykernel_15632\3497561392.py:82: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strate

999
⚠️ Event ends beyond time series: 975-1004 → truncating

✅ Saved final FC array: (15, 48, 35778) → ../4_results/1_arousal/5_Sherlock/Sherlock_FC_array.npy
